# Content-Based Music Recommendation System Using K-Means Clustering

**Authors:** T. Prasanna (R23EF277), Tanishk (R23EF278), Thomson Sunny (R23EF284)  
**Institution:** School of CSE, REVA University, Bengaluru  
**Course:** Machine Learning Applications (MLA)  

---

## Abstract
This notebook implements a content-based music recommendation system using K-Means clustering. Songs are grouped based on intrinsic audio features — danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, and tempo. When a user selects a track, the system recommends acoustically similar songs from the same cluster using nearest-neighbour search, overcoming the cold-start problem without relying on user history.

---

## Pipeline
```
Dataset (Spotify CSV)
    ↓  Data Loading & Preprocessing
    ↓  Feature Extraction & Normalization (StandardScaler)
    ↓  Exploratory Data Analysis (EDA)
    ↓  PCA Dimensionality Reduction (9D → 2D)
    ↓  Elbow Method (optimal k)
    ↓  K-Means Clustering (k=3)
    ↓  Cluster Analysis & Validation
    ↓  Recommendation via Nearest Neighbours
```

## Step 1 — Install & Import Libraries

In [ ]:
# Install required libraries (only needed on Google Colab)
!pip install pandas numpy matplotlib seaborn scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

# Plot style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('All libraries imported successfully.')

## Step 2 — Load Dataset

In [ ]:
# ── Google Colab: upload your Spotify CSV ──────────────────────────────────
try:
    from google.colab import files
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
    print(f'Uploaded: {file_name}')
except ImportError:
    # ── Local: set path to your CSV ───────────────────────────────────────
    file_name = 'spotify_tracks.csv'   # change to your file name
    print(f'Running locally. Using: {file_name}')

In [ ]:
df = pd.read_csv(file_name)

print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
df.head()

## Step 3 — Data Preprocessing

In [ ]:
# Audio features used in the paper
FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

# Check which feature columns exist in the dataset
available = [f for f in FEATURES if f in df.columns]
missing   = [f for f in FEATURES if f not in df.columns]

print(f'Available features : {available}')
if missing:
    print(f'Missing features   : {missing}')
    print('These will be skipped.')

FEATURES = available

In [ ]:
# ── 3.1 Check missing values ───────────────────────────────────────────────
print('Missing values per feature before cleaning:')
print(df[FEATURES].isnull().sum())

# Drop rows with nulls in feature columns
df = df.dropna(subset=FEATURES)
df = df.drop_duplicates()
df = df.reset_index(drop=True)

print(f'\nRows after cleaning: {len(df)}')

In [ ]:
# ── 3.2 Feature matrix ─────────────────────────────────────────────────────
X_raw = df[FEATURES].values

# ── 3.3 Standardisation (zero mean, unit variance) ─────────────────────────
# Paper: "standardization with StandardScaler is applied to transform all
# selected features to have zero mean and unit variance"
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f'Feature matrix shape : {X_scaled.shape}')
print(f'Mean after scaling   : {X_scaled.mean(axis=0).round(4)}')
print(f'Std  after scaling   : {X_scaled.std(axis=0).round(4)}')

## Step 4 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 4.1 Feature Distributions (Fig 1 / Fig 6 in paper) ────────────────────
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, feature in enumerate(FEATURES):
    axes[i].hist(df[feature], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(feature.capitalize(), fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

plt.suptitle('Fig 1: Audio Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature distribution analysis:')
print(df[FEATURES].describe().round(3))

In [ ]:
# ── 4.2 Correlation Heatmap (Fig 7 in paper) ──────────────────────────────
# Paper finding: energy ↔ loudness (strong +ve), acousticness ↔ energy (strong -ve)
plt.figure(figsize=(11, 8))
corr = df[FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('Fig 2: Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — Dimensionality Reduction with PCA

In [ ]:
# Paper: "nine-dimensional audio feature space is projected onto two
# principal components, enabling visualization in a 2D plane"
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_ * 100
print(f'PC1 explains: {explained[0]:.2f}%')
print(f'PC2 explains: {explained[1]:.2f}%')
print(f'Total variance explained: {sum(explained):.2f}%')

## Step 6 — Elbow Method (Optimal k)

In [ ]:
# Paper: "Elbow Method is applied by computing inertia for different values of k"
# Paper finding: inertia drops sharply until k=3, then marginal improvement

inertias = []
K_range  = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', linewidth=1.5, label='Optimal k=3')
plt.fill_between(K_range, inertias, alpha=0.08, color='blue')
plt.xlabel('Number of Clusters (k)', fontsize=13)
plt.ylabel('Inertia (Sum of Squared Distances)', fontsize=13)
plt.title('Fig 3: Elbow Method for Optimal k', fontsize=15, fontweight='bold')
plt.legend(fontsize=12)
plt.xticks(K_range)
plt.tight_layout()
plt.savefig('elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()

print('Inertia values:')
for k, inertia in zip(K_range, inertias):
    print(f'  k={k:2d}  →  {inertia:.2f}')

## Step 7 — K-Means Clustering (k=3)

In [ ]:
# Paper: "k=3 is selected based on experimentation and the need to
# balance interpretability with musical diversity"
K = 3

kmeans = KMeans(n_clusters=K, init='k-means++', n_init=10,
                max_iter=300, random_state=42)
kmeans.fit(X_scaled)

df['cluster'] = kmeans.labels_

print(f'K-Means trained with k={K}')
print(f'Inertia: {kmeans.inertia_:.4f}')
print(f'\nCluster sizes:')
print(df['cluster'].value_counts().sort_index())

In [ ]:
# ── Silhouette Score ───────────────────────────────────────────────────────
sil_score = silhouette_score(X_scaled, kmeans.labels_, sample_size=5000, random_state=42)
print(f'Silhouette Score: {sil_score:.4f}')
print('(Score > 0.25 indicates meaningful cluster structure)')

## Step 8 — PCA Scatter Plot with Clusters

In [ ]:
# Paper: "Cluster 0: energetic, high tempo & loudness
#         Cluster 1: acoustic, low-energy, high acousticness
#         Cluster 2: balanced, mixed/transitional"

CLUSTER_LABELS = {
    0: 'Energetic / Electronic',
    1: 'Acoustic / Calm',
    2: 'Balanced / Mixed'
}
CLUSTER_COLORS = ['#e74c3c', '#2ecc71', '#3498db']

plt.figure(figsize=(11, 7))
for cluster_id in range(K):
    mask = df['cluster'] == cluster_id
    plt.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=CLUSTER_COLORS[cluster_id],
        label=f'Cluster {cluster_id}: {CLUSTER_LABELS[cluster_id]}',
        alpha=0.5, s=20, linewidths=0
    )

# Plot centroids projected onto PCA space
centroids_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='black', marker='X', s=200, zorder=5, label='Centroids'
)

plt.xlabel(f'PC1 ({explained[0]:.1f}% variance)', fontsize=12)
plt.ylabel(f'PC2 ({explained[1]:.1f}% variance)', fontsize=12)
plt.title('Fig 4: PCA Scatter Plot of Songs (k=3)', fontsize=15, fontweight='bold')
plt.legend(fontsize=11, markerscale=2)
plt.tight_layout()
plt.savefig('pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — Cluster Analysis

In [ ]:
# ── Average feature values per cluster ───────────────────────────────────
cluster_means = df.groupby('cluster')[FEATURES].mean().round(4)
print('Average Audio Features per Cluster:')
print(cluster_means.T.to_string())

In [ ]:
# ── Radar / Bar chart of cluster profiles ─────────────────────────────────
fig, axes = plt.subplots(1, K, figsize=(18, 5), sharey=False)

for cluster_id in range(K):
    vals = cluster_means.loc[cluster_id]
    axes[cluster_id].bar(FEATURES, vals, color=CLUSTER_COLORS[cluster_id], alpha=0.8, edgecolor='white')
    axes[cluster_id].set_title(f'Cluster {cluster_id}\n{CLUSTER_LABELS[cluster_id]}',
                               fontsize=12, fontweight='bold')
    axes[cluster_id].set_xticklabels(FEATURES, rotation=45, ha='right', fontsize=9)
    axes[cluster_id].set_ylabel('Mean Value')

plt.suptitle('Fig 5: Cluster Audio Profiles', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10 — Music Recommendation System

In [ ]:
# Paper: "Nearest Neighbors search is performed within the selected cluster.
# System identifies songs whose feature vectors have the smallest Euclidean
# distance to the input track, capturing subtle similarities in rhythm,
# mood, and structure."

def get_recommendations(track_name, df, X_scaled, kmeans, scaler,
                         features=FEATURES, n_recommendations=10,
                         name_col=None):
    """
    Recommend songs similar to the given track using K-Means cluster
    + Nearest Neighbour search within the cluster.
    """
    # Auto-detect name column
    if name_col is None:
        for col in ['track_name', 'name', 'title', 'song_name', 'track']:
            if col in df.columns:
                name_col = col
                break

    if name_col is None:
        print('No track name column found. Please specify name_col.')
        return

    # Find track (case-insensitive partial match)
    matches = df[df[name_col].str.lower().str.contains(track_name.lower(), na=False)]

    if matches.empty:
        print(f'Track "{track_name}" not found in dataset.')
        print(f'Sample tracks: {df[name_col].dropna().sample(5).tolist()}')
        return

    # Use first match
    track_idx   = matches.index[0]
    track_row   = df.loc[track_idx]
    track_cluster = int(track_row['cluster'])

    print(f'\n🎵 Input Track   : {track_row[name_col]}')
    print(f'📂 Cluster       : {track_cluster} — {CLUSTER_LABELS[track_cluster]}')
    print(f'\nAudio Features:')
    for f in features:
        print(f'  {f:20s}: {track_row[f]:.4f}')

    # Get all songs in the same cluster
    cluster_mask    = df['cluster'] == track_cluster
    cluster_indices = df.index[cluster_mask].tolist()
    X_cluster       = X_scaled[cluster_mask]

    # Nearest Neighbours within cluster
    nn = NearestNeighbors(n_neighbors=min(n_recommendations + 1, len(cluster_indices)),
                           metric='euclidean')
    nn.fit(X_cluster)

    track_scaled = X_scaled[track_idx].reshape(1, -1)
    distances, indices = nn.kneighbors(track_scaled)

    # Map back to original dataframe indices
    rec_indices = [cluster_indices[i] for i in indices[0] if cluster_indices[i] != track_idx]
    rec_df      = df.loc[rec_indices[:n_recommendations]].copy()

    print(f'\n🎧 Top {n_recommendations} Recommendations from Cluster {track_cluster}:')
    print('─' * 60)

    display_cols = [name_col]
    for col in ['artists', 'artist_name', 'artist', 'album_name']:
        if col in rec_df.columns:
            display_cols.append(col)
            break
    display_cols += ['danceability', 'energy', 'valence', 'tempo']
    display_cols  = [c for c in display_cols if c in rec_df.columns]

    print(rec_df[display_cols].to_string(index=False))
    return rec_df

print('Recommendation function ready.')

In [ ]:
# ── Try it out ─────────────────────────────────────────────────────────────
# Change 'Shape of You' to any song name in your dataset

recommendations = get_recommendations(
    track_name='Shape of You',
    df=df,
    X_scaled=X_scaled,
    kmeans=kmeans,
    scaler=scaler,
    n_recommendations=10
)

In [ ]:
# ── Visualise the recommendation (Fig 8 in paper) ─────────────────────────
if recommendations is not None and not recommendations.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    plot_features = ['danceability', 'energy', 'valence', 'acousticness', 'tempo']
    plot_features = [f for f in plot_features if f in recommendations.columns]

    # Normalise tempo to 0-1 for fair comparison
    rec_plot = recommendations[plot_features].copy()
    if 'tempo' in rec_plot.columns:
        rec_plot['tempo'] = rec_plot['tempo'] / 200

    rec_plot.T.plot(kind='bar', ax=axes[0], legend=False, colormap='tab10', alpha=0.8)
    axes[0].set_title('Fig 6: Feature Profile of Recommended Songs', fontweight='bold')
    axes[0].set_xticklabels(plot_features, rotation=30, ha='right')
    axes[0].set_ylabel('Normalised Value')

    # Scatter: energy vs valence for recommended songs in PCA space
    all_cluster_mask = df['cluster'] == recommendations['cluster'].iloc[0]
    axes[1].scatter(X_pca[all_cluster_mask, 0], X_pca[all_cluster_mask, 1],
                    c='lightgrey', s=15, label='Same cluster')
    rec_idx = recommendations.index
    axes[1].scatter(X_pca[rec_idx, 0], X_pca[rec_idx, 1],
                    c='red', s=60, zorder=5, label='Recommended')
    axes[1].set_title('Fig 7: Recommendations in PCA Space', fontweight='bold')
    axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('recommendation_output.png', dpi=150, bbox_inches='tight')
    plt.show()

## Step 11 — Results & Evaluation Summary

In [ ]:
print('=' * 55)
print('      MODEL EVALUATION SUMMARY')
print('=' * 55)
print(f'  Algorithm          : K-Means Clustering')
print(f'  Number of Clusters : k = {K}')
print(f'  Features Used      : {len(FEATURES)}')
print(f'  Dataset Size       : {len(df)} songs')
print(f'  Inertia            : {kmeans.inertia_:.2f}')
print(f'  Silhouette Score   : {sil_score:.4f}')
print('─' * 55)
print('  Cluster Breakdown:')
for cid, label in CLUSTER_LABELS.items():
    count = (df['cluster'] == cid).sum()
    pct   = count / len(df) * 100
    print(f'    Cluster {cid} ({label}): {count} songs ({pct:.1f}%)')
print('─' * 55)
print('  Key Findings (Paper):')
print('    - Energy strongly correlates with Loudness (+ve)')
print('    - Acousticness negatively correlates with Energy')
print('    - Danceability correlates with Valence and Tempo')
print('    - Elbow point confirmed at k=3')
print('    - Cold-start problem solved (no user data needed)')
print('=' * 55)

## Conclusion

This notebook implements a complete **content-based music recommendation system** using K-Means clustering as described in the project paper:

| Step | Description |
|------|-------------|
| Preprocessing | Null removal, deduplication, StandardScaler normalization |
| Feature Extraction | 9 Spotify audio features per track |
| EDA | Feature distributions + correlation heatmap |
| PCA | 9D → 2D for visualization |
| Elbow Method | Inertia plotted for k=1..10, elbow at k=3 |
| K-Means | k=3 clusters — Energetic, Acoustic, Balanced |
| Recommendation | Nearest Neighbour search within cluster |

**Future Scope:** Hybrid models (collaborative + content), deep learning feature extraction (CNNs on spectrograms), Spotify API integration for real-time recommendations.